# 03. 名寄せと結合

02章で整えた34行を、**店舗マスタ・商品マスタに突き合わせる**のがこの章です。

## この章のゴール

```
  clean 34行   店名は「みなとストア渋谷店」「ミナトストア 渋谷」「みなと渋谷」のまま
     ↓   別名辞書で shop_cd に寄せる (名寄せ)
     ↓   店舗マスタ・商品マスタを結合する
  fact  34行   shop_cd / category が付いた。行数は変わらない
```

**行数が変わらないことが、この章の合格条件です。**
結合は、うっかりすると黙って行が増えます。それを防ぐ書き方を身につけます。

| 名前 | 中身 | 作る章 |
| --- | --- | --- |
| `clean` | 表記と型を整えた34行 | 02章 |
| `fact` | `clean` にマスタを結合したもの。**除外はまだしない** | **03章(この章)** |
| `target` | `fact` から集計対象外を除いたもの | 04章 |

> この章では**1行も除外しません。** テスト伝票も閉店した店の売上も、
> `fact` に入ったままにしておきます。何を落とすかは04章で決めます。
> **「見つける」と「落とす」を別の章に分けてある**のは、意図的です。

## 01章で保留したこと

01章の最後で、重複チェックをして「0件」という結果になりました。
そのとき、こう書いてあります。

> **重複チェックは、表記を整えたあとでなければ意味を持ちません。**
> 整える(02章)→ 名寄せする(03章)→ そのあとでもう一度この確認に戻ってきます。

**この章の7節が、その「戻ってくる」ところです。**

---
## 0. 前章までのまとめ

**次のセルは01章と02章の答えです。読み飛ばして実行してかまいません。**

In [ ]:
import unicodedata

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

# ---------------- 01章: 取り込み ----------------
COLUMNS = ["sale_date", "shop_name", "item_cd", "qty", "amount",
           "tax_type", "note", "source"]

RENAME_STORE = {"売上日": "sale_date", "店舗名": "shop_name", "商品CD": "item_cd",
                "数量": "qty", "金額": "amount", "備考": "note"}
RENAME_EC = {"受注日": "sale_date", "店舗名": "shop_name", "商品CD": "item_cd",
             "数量": "qty", "金額税抜": "amount", "ステータス": "note"}

SOURCES = [
    {"path": "/data/sales_2024-04_old.csv", "encoding": "cp932",
     "tax_type": "税込", "source": "old", "rename": RENAME_STORE},
    {"path": "/data/sales_2024-04_new.csv", "encoding": "utf-8",
     "tax_type": "税込", "source": "new", "rename": RENAME_STORE},
    {"path": "/data/sales_2024-04_ec.csv", "encoding": "utf-8",
     "tax_type": "税抜", "source": "ec", "rename": RENAME_EC},
]


def read_one(spec):
    df = pd.read_csv(spec["path"], dtype=str, keep_default_na=False,
                     encoding=spec["encoding"])
    df = df.rename(columns=spec["rename"])
    df = df.assign(tax_type=spec["tax_type"], source=spec["source"])
    return df[COLUMNS]


def load_raw(sources=SOURCES):
    df = pd.concat([read_one(s) for s in sources], ignore_index=True)
    print(f"取り込み: {len(df)}行  内訳 {df['source'].value_counts().to_dict()}")
    return df


# ---------------- 02章: クレンジング ----------------
NA_TOKENS = ["", "-", "N/A"]
TEXT_COLUMNS = ["sale_date", "shop_name", "item_cd", "qty", "amount", "note"]
FORMATS = ["%Y/%m/%d", "%Y年%m月%d日", "%Y-%m-%d"]
TAX_RATE = 1.1


def norm(s):
    """NFKC正規化して、前後の空白を落とす。"""
    return unicodedata.normalize("NFKC", s).strip()


def parse_date(s):
    d = pd.to_datetime(s, format=FORMATS[0], errors="coerce")
    for fmt in FORMATS[1:]:
        d = d.fillna(pd.to_datetime(s, format=fmt, errors="coerce"))
    return d


def clean_raw(raw):
    """raw を整形して (clean, rejected) に分ける。行は1つも捨てない。"""
    df = raw.copy()
    for c in TEXT_COLUMNS:
        df[c] = df[c].map(norm)
    df = df.replace(NA_TOKENS, pd.NA)
    df["item_cd"] = df["item_cd"].str.zfill(4)
    df["amount"] = df["amount"].str.replace(r"[¥,]", "", regex=True)
    df["sale_date"] = parse_date(df["sale_date"])
    df["qty"] = pd.to_numeric(df["qty"], errors="coerce").astype("Int64")
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce").astype("Int64")
    is_excl = df["tax_type"] == "税抜"
    df["amount_incl"] = df["amount"].where(
        ~is_excl, (df["amount"] * TAX_RATE).round()).astype("Int64")
    ok = df["sale_date"].notna() & df["qty"].notna() & df["amount"].notna()
    clean = df[ok].reset_index(drop=True)
    rejected = df[~ok].reset_index(drop=True)
    rate = len(rejected) / len(df) * 100
    print(f"クレンジング: {len(df)}行 → clean {len(clean)}行 / "
          f"rejected {len(rejected)}行 ({rate:.1f}%)")
    return clean, rejected


raw = load_raw()
clean, rejected = clean_raw(raw)
clean.head(3)

---
## 1. 名寄せをしないと、何が起きるか

先に、やらなかった場合を見ておきます。**店名ごとに売上を集計してみます。**

In [ ]:
by_name = clean.groupby("shop_name")["amount_incl"].agg(["count", "sum"])
by_name

**8行出ました。みなとストアは3店舗のはずです。**

```
みなとストア渋谷店     ┐
ミナトストア 渋谷      ├─ 全部おなじ渋谷店
みなと渋谷            ┘

みなとストア新宿店        ┐
(株)みなとストア 新宿店   ├─ 全部おなじ新宿店
ミナトストア 新宿         ┘

みなとストア横浜店   ┐
みなと横浜          ┘─ おなじ横浜店
```

このまま「店舗別売上」として出したら、**渋谷店の売上が3つに割れた表**が出ていきます。
合計は合っているので、**合計だけ見ている限り気づけません。**

02章の NFKC は、この問題を解決しませんでした。
`ミナトストア　渋谷` → `ミナトストア 渋谷` と空白が半角になっただけです。

**「ミナトストア 渋谷」と「みなとストア渋谷店」が同じ店だ、というのは
Unicode の知識ではなく、業務の知識です。** 機械には決められません。
人間が対応表を作って渡す必要があります。それが**名寄せ**です。

In [ ]:
# ✍ 書いてみる: clean に店名が何種類あるか数えてください。

ans = ...   # ここに書く

assert ans == 8, f"8種類のはずです: {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = clean["shop_name"].nunique()
```

</details>

---
## 2. 別名辞書を使う

対応表は用意されています。`data/shop_alias.csv` です。

In [ ]:
print(open("/data/shop_alias.csv", encoding="utf-8").read())

「この表記が来たら、この店舗コード」という対応が9行です。
**表記の数だけ行がある**のがこの形の特徴で、新しい表記が出てきたら1行足します。

`S04`(大宮店)まで入っているのに注目してください。
**閉店した店の表記も辞書には残しておきます。** 過去のデータを読み直せなくなるからです。

### 2-1. まず、そのまま結合してみる

対応表を持ってきて `merge` します。**うまくいかない**ので、その様子を見ます。

In [ ]:
alias_raw = pd.read_csv("/data/shop_alias.csv", dtype=str, keep_default_na=False)

test = clean.merge(alias_raw, left_on="shop_name", right_on="alias",
                   how="left", indicator=True)

print(test["_merge"].value_counts().to_dict())
print()
print("マッチしなかった店名:")
print(test.loc[test["_merge"] == "left_only", "shop_name"].value_counts())

**14行がマッチしませんでした。** 旧POSの2つの表記です。

原因は空白です。

```
clean      'ミナトストア 渋谷'    ← 02章の NFKC で半角空白になっている
alias_raw  'ミナトストア　渋谷'    ← CSVから読んだまま。全角空白
```

**辞書の側を正規化していないので、同じ文字列になっていません。**

> `indicator=True` を付けると `_merge` 列が付き、
> `both` / `left_only` / `right_only` のどれかが入ります。
> **結合の結果を確かめるいちばん簡単な方法**なので、まず付けて確認し、
> 確認が済んだら外す、という使い方をします。

### 2-2. 辞書の側にも同じ正規化をかける

**突き合わせる2つの値には、同じ処理を通します。** 片側だけでは揃いません。

In [ ]:
alias = pd.read_csv("/data/shop_alias.csv", dtype=str, keep_default_na=False)
alias["alias"] = alias["alias"].map(norm)   # 02章で定義した norm を再利用

test = clean.merge(alias, left_on="shop_name", right_on="alias",
                   how="left", indicator=True)

print(test["_merge"].value_counts().to_dict())
print()
print(test["shop_cd"].value_counts().sort_index())

**34行すべてマッチしました。**

| shop_cd | 店 | 行数 |
| --- | --- | --- |
| S01 | 渋谷店 | 15 |
| S02 | 新宿店 | 7 |
| S03 | 横浜店 | 11 |
| S04 | 大宮店 | **1** |

3店舗に寄りました。そして **`S04`(閉店したはずの大宮店)が1行**あります。
これは3節で扱います。

> 名寄せの実装で覚えておくことは1つだけです。
> **「両側に同じ正規化関数を通す」。** 02章で作った `norm` をそのまま使う、という形にしておくと、
> 片側だけ直して壊す、という事故が起きません。

In [ ]:
# ✍ 書いてみる: 正規化した辞書 alias で、"ミナトストア 渋谷" を引いて shop_cd を出してください。
#              (ヒント: alias から alias 列が一致する行を絞って shop_cd を取ります)

ans = ...   # ここに書く

assert ans == "S01", f"'S01' のはずです: {ans!r}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = alias.loc[alias["alias"] == "ミナトストア 渋谷", "shop_cd"].iloc[0]
```

</details>

---
## 3. 結合で行が増える事故を防ぐ

結合はデータ処理でいちばん事故が多いところです。
**行が増えても減っても、エラーは出ません。** 数字が静かに狂います。

### 3-1. 行が増えるのはどういうときか

辞書の側に**同じキーが2行ある**と増えます。実験してみます。

In [ ]:
# わざと壊した辞書。渋谷店の行が2つある (S01 と S99)
broken = pd.concat([
    alias,
    pd.DataFrame({"alias": ["みなとストア渋谷店"], "shop_cd": ["S99"]}),
], ignore_index=True)

bad = clean.merge(broken, left_on="shop_name", right_on="alias", how="left")

print(f"元:     {len(clean)}行")
print(f"結合後: {len(bad)}行")
print(f"金額合計 {clean['amount_incl'].sum():,} → {bad['amount_incl'].sum():,}")

**行が増え、金額も増えました。**

`みなとストア渋谷店` の売上が、`S01` と `S99` の2行に複製されたからです。
これは「二重計上」で、集計の間違いのなかで**いちばん見つけにくい部類**です。

- エラーは出ません
- 合計だけ見ると「ちょっと増えたかな」くらいにしか見えません
- 気づくのは、たいてい誰かが「先月より売上が多すぎる」と言い出したときです

原因は**結合のコードではなく、辞書のほう**にあります。
コードを何度読み直しても見つかりません。

### 3-2. `validate` で止める

`merge` は「キーの関係がどうあるべきか」を宣言できます。

| 指定 | 意味 |
| --- | --- |
| `"m:1"` | 左は重複してよい。**右のキーは一意でなければならない** |
| `"1:1"` | 両方一意 |
| `"1:m"` | 左が一意 |

明細にマスタを付ける結合は、ほぼ必ず **`m:1`** です。

In [ ]:
try:
    clean.merge(broken, left_on="shop_name", right_on="alias",
                how="left", validate="m:1")
except Exception as e:
    print(f"{type(e).__name__}: {e}")

**例外で止まりました。** 黙って増えるより、ずっとよい状態です。

`validate` は「こうなっているはずだ」という**思い込みを、コードに書いておく道具**です。
思い込みが外れたときに、その場で止まります。

### 結合するときの三点セット

```python
df.merge(master, on="key", how="left", validate="m:1")
```

| | なぜ |
| --- | --- |
| `how="left"` | **左の行を落とさない。** `inner` だとマスタに無い行が黙って消える |
| `validate="m:1"` | **右の重複で行が増えるのを止める** |
| `indicator=True` | マッチしなかった行を数える(確認が済んだら外す) |

この3つを毎回書く、と決めておけば、結合の事故はほとんど防げます。

In [ ]:
# ✍ 書いてみる: 壊れた辞書 broken に、重複しているキーがいくつあるか数えてください。
#              (ヒント: alias 列の duplicated)

ans = ...   # ここに書く

assert ans == 1, f"1件のはずです: {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = broken["alias"].duplicated().sum()
```

</details>

**マスタを受け取ったら、まずキーの重複を数える。** これも習慣にしておくと安全です。
`validate` で止まってから調べるより早く済みます。

---
## 4. 店舗マスタを結合する

`shop_cd` が付いたので、店舗マスタから正式名称とエリアを取ります。

In [ ]:
shops = pd.read_csv("/data/shops.csv", dtype=str, keep_default_na=False)
shops = shops.replace("", pd.NA)
shops["close_date"] = pd.to_datetime(shops["close_date"])
shops

4店舗あり、**大宮店(S04)には `close_date` が入っています。2024-03-31 に閉店**しています。

`replace("", pd.NA)` を通しているのは、空欄を欠損として扱うためです。
02章でやったことと同じで、**マスタもデータなので、同じように整えます。**

### 4-1. 同じ名前の列がぶつかる

`clean` にも `shops` にも `shop_name` があります。そのまま結合するとどうなるか見てみます。

In [ ]:
crash = clean.merge(alias, left_on="shop_name", right_on="alias", how="left",
                    validate="m:1").merge(shops, on="shop_cd", how="left", validate="m:1")

print([c for c in crash.columns if "shop_name" in c])

**`shop_name_x` と `shop_name_y` になりました。**

pandas は勝手に消したりはせず、`_x`(左)と `_y`(右)を付けて両方残します。
親切ではあるのですが、**このまま先に進むと、どちらがどちらか分からなくなります。**

`_x` や `_y` が出てきたら、**そのまま使わずに名前を決め直してください。**

In [ ]:
fact = clean.rename(columns={"shop_name": "shop_name_raw"})

fact = fact.merge(alias, left_on="shop_name_raw", right_on="alias",
                  how="left", validate="m:1").drop(columns="alias")
fact = fact.merge(shops[["shop_cd", "shop_name", "area", "close_date"]],
                  on="shop_cd", how="left", validate="m:1")

print(f"{len(clean)}行 → {len(fact)}行")
fact[["sale_date", "shop_name_raw", "shop_cd", "shop_name", "area", "amount_incl"]].head()

結合前の店名を `shop_name_raw` に改名してから結合しました。これで

- `shop_name_raw` … データに書かれていた表記
- `shop_name` … マスタの正式名称

と、意味が名前で分かります。

**元の表記を残しておくのは、名寄せがうまくいかなかったときに調べるため**です。
`shop_cd` だけ残して捨ててしまうと、「どの表記が来て失敗したのか」を追えません。

### 4-2. 閉店した店の売上を見つける

`close_date` が使えるようになりました。**閉店日より後の売上**を探します。

In [ ]:
closed = fact[fact["close_date"].notna() & (fact["sale_date"] > fact["close_date"])]

print(f"閉店後の売上: {len(closed)}行  {closed['amount_incl'].sum():,}円")
closed[["sale_date", "shop_name", "close_date", "item_cd", "qty", "amount_incl", "source"]]

**1行見つかりました。** 大宮店は 2024-03-31 に閉店しているのに、4/10 に450円の売上が立っています。

考えられる理由はいくつもあります。

- 閉店処理が終わっていない端末から送られてきた
- テスト用の伝票が本番データに混ざった
- 店舗コードの打ち間違い
- マスタの `close_date` のほうが間違っている

**どれが正しいかは、データからは分かりません。** 業務側に聞くしかない種類の問題です。

だからこの章では**見つけるところまで**にして、落としません。
落とすかどうかは04章で決めます。

> **「検知」と「対処」を分けておく**と、対処の方針が変わったときにコードを直す場所が1箇所で済みます。
> 見つけた瞬間に `drop` してしまうと、あとから「やっぱり残したい」と言われたときに困ります。

In [ ]:
# ✍ 書いてみる: shop_cd ごとの売上合計を、金額の大きい順で出してください。

ans = ...   # ここに書く

assert ans.index.tolist() == ["S03", "S01", "S02", "S04"], ans.index.tolist()
assert ans["S04"] == 450, f"S04 は450円のはずです: {ans['S04']}"
print("OK")
print(ans)

<details>
<summary>答え</summary>

```python
ans = fact.groupby("shop_cd")["amount_incl"].sum().sort_values(ascending=False)
```

</details>

1節の8行が4行になりました。**渋谷店の売上が1つにまとまっています。**
これが名寄せの成果です。

いちばん売れているのは横浜店(S03)で 9,759円。渋谷店(S01)が 8,040円と続きます。
1節の表では、渋谷店の売上が3行に割れていたので、**この順位は出せませんでした。**
名寄せをして初めて「どの店がいちばん売れたか」が言えるようになります。

---
## 5. 商品マスタを結合する

同じやり方で商品マスタも結合します。今度は**マッチしない行が出ます。**

In [ ]:
items = pd.read_csv("/data/items.csv", dtype=str, keep_default_na=False)
items = items.replace("", pd.NA)
items["discontinued_date"] = pd.to_datetime(items["discontinued_date"])
items

6商品です。ここでも2つ、気になるところがあります。

- **`0006 スープ` の `category` が空欄**です
- **`0005 プリン` に `discontinued_date`(廃番日)** が入っています

どちらもあとで扱います。まず結合します。

In [ ]:
fact = fact.merge(items[["item_cd", "item_name", "category", "discontinued_date"]],
                  on="item_cd", how="left", validate="m:1", indicator=True)

print(fact["_merge"].value_counts().to_dict())
print()
fact.loc[fact["_merge"] == "left_only",
         ["sale_date", "shop_name", "item_cd", "qty", "amount_incl", "source"]]

**1行マッチしませんでした。** 商品コード `0099` です。マスタに存在しません。

`how="left"` にしてあるので、**この行は消えずに残っています。**
`item_name` と `category` が欠損になっているだけです。

もし `how="inner"` で結合していたら、この700円は**黙って消えていました。**
合計も減りますが、減ったことに気づく手がかりはどこにも残りません。

> **`inner join` は、静かに行を捨てる操作です。**
> マスタとの結合では、まず `left` を選んでください。
> `inner` を使ってよいのは、「マッチしない行は本当に要らない」と確認できたときだけです。

In [ ]:
fact = fact.drop(columns="_merge")

# ✍ 書いてみる: item_name が欠損している行の金額合計を出してください。

ans = ...   # ここに書く

assert ans == 700, f"700円のはずです: {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = fact.loc[fact["item_name"].isna(), "amount_incl"].sum()
```

</details>

### 5-1. カテゴリの欠損を埋める

カテゴリが欠けている行は2種類あります。

In [ ]:
print(fact["category"].value_counts(dropna=False))
print()
print(fact.loc[fact["category"].isna(),
               ["item_cd", "item_name", "qty", "amount_incl", "source"]])

合わせて4行が欠損です。中身は2種類あります。

| 商品 | なぜ欠損か |
| --- | --- |
| `0006 スープ` (3行) | **マスタにはあるが、カテゴリ欄が空**。登録漏れ |
| `0099` (1行) | **マスタに無い**ので、そもそも取れない |

### 欠損が `<NA>` と `NaN` に分かれています

`value_counts` の出力をよく見てください。欠損が**2行に分かれています**。

```
<NA>     3      ← マスタの空欄。pd.NA (自分で replace("", pd.NA) した)
NaN      1      ← 結合でマッチしなかった穴。np.nan (pandas が入れた)
```

**どちらも「欠損」ですが、中身は別のオブジェクトです。**
読み物 `docs/03` に「`NaN` は3種類ある(`None` / `float('nan')` / `pd.NaT`)」と書いてあるのは、
この話です。

だから **`== pd.NA` のような比較で欠損を探してはいけません。** `isna()` を使えば、
どの種類でもまとめて拾えます。上のセルで `isna()` を使っているのは、そのためです。

> 見た目が同じで正体が違う、というのは追いにくい種類のバグです。
> **欠損の判定は必ず `isna()` / `notna()` で行う**、と決めておいてください。

原因は違いますが、**サマリを見る人にとっては同じ**です。
どちらも「カテゴリが分からない売上」なので、`未分類` にまとめます。
`fillna` は種類を問わず埋めてくれるので、2つを区別せずに書けます。

In [ ]:
fact["category"] = fact["category"].fillna("未分類")

print(fact.groupby("category")["amount_incl"].agg(["count", "sum"]))

**`未分類` に4行・4,059円**が集まりました。全体の15%ほどです。無視できる大きさではありません。

ここで大事なのは、**`未分類` という行を表に出すこと**です。
消してしまうと、見る人は「カテゴリの合計 = 全体の合計」だと思い込みます。
`未分類` が見えていれば、「これは何だろう」と聞いてもらえます。

> 欠損を埋めるときは、**「埋めた」と分かる値**にしてください。
> `その他` のような既存カテゴリに混ぜたり、`0` で埋めたりすると、
> **埋めた事実そのものが消えます。**

### 5-2. 廃番日を確かめる

`0005 プリン` は 2024-04-05 に廃番になっています。それより後に売れていないか見ます。

In [ ]:
sold = fact[fact["item_cd"] == "0005"]
print(sold[["sale_date", "item_name", "discontinued_date", "qty", "amount_incl"]])

after = fact["discontinued_date"].notna() & (fact["sale_date"] > fact["discontinued_date"])
same = fact["discontinued_date"].notna() & (fact["sale_date"] == fact["discontinued_date"])
print()
print(f"廃番日より後に売れた: {after.sum()}行")
print(f"廃番日ちょうどに売れた: {same.sum()}行")

**廃番日ちょうどに1行**、後は0行でした。問題なしです。

ただし、これは **`>` と `>=` のどちらを書いたかで結論が変わります。**

```
sale_date >  discontinued_date   →  0行。「廃番日当日はまだ売ってよい」
sale_date >= discontinued_date   →  1行。「廃番日から売ってはいけない」
```

**どちらが正しいかは、業務が決めることです。**
「廃番日」が「その日を最後に売らない日」なのか「その日から売れない日」なのかは、
データを見ても分かりません。

境界がぴったり重なる行がデータに1つ入っているのは偶然ではなく、
**この確認をしてもらうため**です。日付の比較を書いたら、
**境界の1件がどちら側に落ちるかを必ず確かめてください。**

In [ ]:
# ✍ 書いてみる: 「廃番日以降 (>=) に売れた」行数を数えてください。

ans = ...   # ここに書く

assert ans == 1, f"1行のはずです: {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = int((fact["discontinued_date"].notna()
           & (fact["sale_date"] >= fact["discontinued_date"])).sum())
```

</details>

---
## 6. 行数と合計が変わっていないことを確かめる

結合を3回やりました。**行数と金額が変わっていないか、ここで確認します。**

In [ ]:
print(f"clean {len(clean)}行  {clean['amount_incl'].sum():,}円")
print(f"fact  {len(fact)}行  {fact['amount_incl'].sum():,}円")

assert len(fact) == len(clean)
assert fact["amount_incl"].sum() == clean["amount_incl"].sum()
print("\n行数も合計も変わっていません")

**34行 / 26,489円のまま**です。02章の最後に「握って次の章に行く」と言った数字です。

結合は**列を増やす操作**であって、行や金額を変える操作ではありません。
変わっていたら、`validate` か `how` のどちらかが間違っています。

> この `assert` を関数の中に入れておくと、あとで誰かが `how="inner"` に変えたときに、
> その場で止まります。**確認を文章ではなくコードで残す**、というのがここの狙いです。

---
## 7. 重複チェックに戻る (01章の宿題)

01章の 4-6 で、こう書きました。

```
key = ["sale_date", "shop_name", "item_cd"]
dup = raw[raw.duplicated(key, keep=False)]
→ 0件
```

そして「**同じものが同じだと判定されていないので、重複のしようがない**」と保留しました。

いまは違います。日付は日付型になり、店名は `shop_cd` に寄り、商品コードは4桁に揃いました。
**同じものが同じだと判定できる状態**です。もう一度やってみます。

In [ ]:
key = ["sale_date", "shop_cd", "item_cd"]
dup = fact[fact.duplicated(key, keep=False)].sort_values(key)

print(f"重複している行数: {len(dup)}")
dup[["sale_date", "shop_cd", "item_cd", "item_name", "qty", "amount_incl", "source"]]

**6行(3組)出ました。** 01章では0件だったものが、整えたら見えるようになりました。

> 01章の「0件」は「重複が無い」ではなく「**判定できていなかった**」でした。
> 表記がバラバラなときの `duplicated` は、**常に0件を返します。**
> 何も見つからなかったときは、「本当に無いのか」「見えていないだけか」を疑ってください。

### この6行は、消すべきでしょうか

中身を見ます。

| 日付 | 店 | 商品 | 数量 | 金額 | 系統 |
| --- | --- | --- | --- | --- | --- |
| 4/2 | S01 | 0001 コーヒー | 3 | 1350 | `old` |
| 4/2 | S01 | 0001 コーヒー | 1 | 450 | `ec` |
| 4/4 | S01 | 0004 サンドイッチ | 1 | 550 | `new` |
| 4/4 | S01 | 0004 サンドイッチ | 1 | 550 | `ec` |
| 4/5 | S03 | 0001 コーヒー | 1 | 450 | `new` |
| 4/5 | S03 | 0001 コーヒー | 3 | 1350 | `ec` |

**3組とも、片方が店頭(`old` / `new`)で、片方が EC です。**

同じ日に、同じ店の商品が、店頭でも EC でも売れた。
**これは二重計上ではなく、別々の売上です。** 消したら売上が減ります。

とくに4/4 の組は、数量も金額も**完全に同じ**です。
機械的に「完全一致は再送」と判断していたら、550円が消えていました。

### 重複を見たときに決めること

`duplicated` が返すのは「**このキーで見ると同じ行**」であって、「同じ売上」ではありません。
出てきたら、次の順で考えます。

| 問い | この6行の場合 |
| --- | --- |
| 1. この表の**1行は何**か | 1回の売上明細 |
| 2. その粒度で**一意になるキー**は何か | **無い**。同じ日に同じ商品が複数回売れてよい |
| 3. だとすると、この重複は何か | **正常。別々の売上** |

**2 で「一意になるキーが無い」なら、重複排除はしません。**

重複排除が要るのは、キーが一意であるべきなのに重複しているときです。
たとえば `order_id` のような伝票番号があれば、それは一意のはずなので、
重複していたら再送か訂正です(読み物 `docs/03` の「重複排除で決めること」)。

**このデータには伝票番号がありません。** だから「1行 = 1明細」として、
そのまま持っておくのが正しい扱いになります。

> ここで `drop_duplicates()` と書いてしまうのは、よくある間違いです。
> **重複を見つけたら消す、ではありません。見つけたら、意味を調べます。**

In [ ]:
# ✍ 書いてみる: この6行のうち、EC 由来の行の金額合計を出してください。

ans = ...   # ここに書く

assert ans == 2350, f"2350円のはずです: {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = dup.loc[dup["source"] == "ec", "amount_incl"].sum()
```

</details>

**2,350円**です。もし機械的に重複排除していたら、
このうちのいくつかが消えていた可能性があります。

**「行が減ったことに説明が付くか」を毎回確かめてください。**

---
## 8. この章のまとめ

3つの結合を**関数1つ**にまとめます。

In [ ]:
def load_masters():
    """3つのマスタを読んで、突き合わせられる形にする。"""
    alias = pd.read_csv("/data/shop_alias.csv", dtype=str, keep_default_na=False)
    alias["alias"] = alias["alias"].map(norm)          # 明細と同じ正規化をかける

    shops = pd.read_csv("/data/shops.csv", dtype=str, keep_default_na=False)
    shops = shops.replace("", pd.NA)
    shops["close_date"] = pd.to_datetime(shops["close_date"])

    items = pd.read_csv("/data/items.csv", dtype=str, keep_default_na=False)
    items = items.replace("", pd.NA)
    items["discontinued_date"] = pd.to_datetime(items["discontinued_date"])
    return alias, shops, items


def join_master(clean):
    """clean にマスタを結合する。行は増やさない。除外もしない。"""
    alias, shops, items = load_masters()

    # 元の表記は shop_name_raw として残す (マスタの shop_name とぶつけない)
    df = clean.rename(columns={"shop_name": "shop_name_raw"})

    df = df.merge(alias, left_on="shop_name_raw", right_on="alias",
                  how="left", validate="m:1").drop(columns="alias")
    df = df.merge(shops[["shop_cd", "shop_name", "area", "close_date"]],
                  on="shop_cd", how="left", validate="m:1")
    df = df.merge(items[["item_cd", "item_name", "category", "discontinued_date"]],
                  on="item_cd", how="left", validate="m:1")

    df["category"] = df["category"].fillna("未分類")

    assert len(df) == len(clean), "結合で行数が変わりました"

    no_shop = df["shop_cd"].isna().sum()
    no_item = df["item_name"].isna().sum()
    closed = (df["close_date"].notna() & (df["sale_date"] > df["close_date"])).sum()
    print(f"結合: {len(df)}行  店舗未マッチ {no_shop} / 商品未マッチ {no_item} / "
          f"閉店後の売上 {closed}")
    return df


fact = join_master(clean)
fact.head()

**関数の中に `assert` と `print` が入っている**のが、この章のまとめの形です。

- `assert len(df) == len(clean)` … 行が増えたら**その場で止まる**
- `print(...)` … 未マッチ件数を**毎回の実行に残す**

02章の `clean_raw` が除外率を出していたのと同じ考え方です。
**異常は「見つけたときに調べる」ものではなく、「毎回数えておいて、変化に気づく」ものです。**

未マッチが 0 → 5 に増えたら、上流で新しい店ができたか、コード体系が変わったかです。
数えていなければ、その変化には気づけません。

In [ ]:
# ✍ 書いてみる: fact が次の3つを満たすか確かめて、True を ans に入れてください。
#              (1) clean と同じ34行  (2) shop_cd に欠損が無い  (3) category に欠損が無い

ans = ...   # ここに書く

assert ans is True
print("OK")

<details>
<summary>答え</summary>

```python
ans = bool(
    len(fact) == len(clean)
    and fact["shop_cd"].notna().all()
    and fact["category"].notna().all()
)
```

</details>

### 04章に渡すもの

In [ ]:
print(f"fact  {len(fact)}行  {fact['amount_incl'].sum():,}円")
print()
print("04章で除外するかどうかを決める材料:")
print(f"  テスト伝票      {(fact['note'].fillna('') == 'テスト').sum()}行")
print(f"  閉店後の売上    {(fact['close_date'].notna() & (fact['sale_date'] > fact['close_date'])).sum()}行")
print(f"  返品            {(fact['note'].fillna('') == '返品').sum()}行")
print(f"  マスタ未登録商品 {fact['item_name'].isna().sum()}行")

**`fact` は34行・26,489円のまま**、02章から1円も変わっていません。

そのうえで、「あとで判断が要るもの」が4種類あることが分かりました。
`fact` には全部入ったままです。**落とすかどうかは04章で決めます。**

---
## この章で分かったこと

| | |
| --- | --- |
| 名寄せ | 表記を業務コードに寄せる。**辞書の側にも同じ正規化を通す** |
| 結合の三点セット | `how="left"` / `validate="m:1"` / `indicator=True` |
| `how="inner"` | マッチしない行を**黙って捨てる**。マスタとの結合では使わない |
| `validate="m:1"` | 右のキーが重複していたら止まる。**二重計上を防ぐ** |
| 列名の衝突 | `_x` `_y` が出たら、そのまま使わず名前を決め直す |
| 元の値 | 名寄せ前の表記(`shop_name_raw`)を残す。失敗を追えるようにするため |
| 欠損の穴埋め | `未分類` のように**埋めたと分かる値**にする。既存の値に混ぜない |
| 日付の境界 | `>` と `>=` で結論が変わる。境界の1件がどちらに落ちるか確かめる |
| 重複 | 見つけても**消さない**。まず「1行は何か」「一意になるキーは何か」を決める |
| 検知と対処 | **見つける章と落とす章を分ける。** 方針が変わっても直す場所が1箇所で済む |

## 次の章に持ち越す宿題

- [ ] テスト伝票(1行)を集計に入れるか決める
- [ ] 閉店した大宮店の売上(1行)を集計に入れるか決める
- [ ] 返品(2行)をどう扱うか決める
- [ ] 売上が無かった日を、サマリにどう出すか決める
- [ ] 集計した結果が `fact` の合計と合っているか検算する

次: `04-aggregate.ipynb`